# SEA — Milestones 2 and 3 on Google Colab

Use a **GPU** runtime: `Runtime → Change runtime type → T4`.

Upload the **code** (this repo without `data/`) and the **datasets** to Drive. Do **not** upload the 33.8 GB MIMIC zip.

| Dataset | Upload this |
|---|---|
| PTB-XL | `ptb-xl-*.zip` or a folder containing `ptbxl_database.csv` |
| Chapman | CinC 2021 WFDB (`JS*.hea` + `.mat`) or Zheng `Diagnostics.xlsx` + ECG CSVs |
| MIMIC-IV-ECG | `record_list.csv` + `machine_measurements.csv` only (~240 MB) |

MIMIC reports are mapped locally. Do not paste a PhysioNet password or ECG reports into chat.

In [ ]:
# True = 2 epochs / 128 records / 1 adaptation rep (wiring check, ~15 min)
# False = full M2 + M3 (several hours on a T4; keep the tab open)
QUICK = False
PREFETCH_MIMIC = True  # download the 8k waveform subset onto Colab disk

DRIVE_DATA = "/content/drive/MyDrive/sea_data"      # zips / extracted datasets
DRIVE_CODE = "/content/drive/MyDrive/sea"           # this repo, or leave unused
DRIVE_RESULTS = "/content/drive/MyDrive/sea_results"
LOCAL_DATA = "/content/data"

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU (T4)"

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys, subprocess, shutil

drive.mount("/content/drive")

# Optional PhysioNet login for MIMIC tables/waveforms. Add secrets named
# PHYSIONET_USER and PHYSIONET_PASSWORD in the Colab key icon (left sidebar).
try:
    os.environ["PHYSIONET_USER"] = userdata.get("PHYSIONET_USER")
    os.environ["PHYSIONET_PASSWORD"] = userdata.get("PHYSIONET_PASSWORD")
    print("PhysioNet secrets loaded (password not printed).")
except Exception:
    print("No PhysioNet secrets. Public PTB-XL/Chapman still run; MIMIC needs secrets if the CSVs are not already uploaded.")

In [ ]:
# Locate the SEA repo: Drive copy, uploaded zip, or this notebook's folder.
CANDIDATES = [
    Path(DRIVE_CODE),
    Path("/content/sea"),
    Path.cwd(),
    Path.cwd().parent,
]
REPO = None
for cand in CANDIDATES:
    if (cand / "src" / "sea").exists() and (cand / "scripts" / "run_milestone2.py").exists():
        REPO = cand.resolve()
        break

if REPO is None:
    zips = list(Path("/content/drive/MyDrive").glob("**/sea*.zip"))[:5]
    print("Repo not found. Upload sea.zip (code only) to Drive or set DRIVE_CODE.")
    print("zip candidates:", zips)
    raise FileNotFoundError("SEA repo not found")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("REPO", REPO)
!pip -q install -e . wfdb tqdm pyyaml scikit-learn scipy pandas openpyxl matplotlib seaborn

In [ ]:
from sea.config import load_config, save_config
from sea.data.prepare import prepare_uploads

src = Path(DRIVE_DATA)
if not src.exists():
    src = Path("/content/drive/MyDrive")
    print("DRIVE_DATA missing; scanning MyDrive for PhysioNet zips/folders.")

cfg = load_config(REPO / "configs" / "colab.yaml").raw
cfg["output_dir"] = DRIVE_RESULTS
Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)
Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)

roots = prepare_uploads(src, Path(LOCAL_DATA), cfg, prefetch_mimic=PREFETCH_MIMIC)
runtime = REPO / "configs" / "runtime.yaml"
save_config(cfg, runtime)
print("runtime config:", runtime)
print("roots:", roots)
assert "ptbxl" in roots, "PTB-XL not found. Put the zip or extracted folder in Drive and set DRIVE_DATA."

## Milestone 2

Train `resnet1d_wang` on PTB-XL diagnostic superclasses (NORM / MI / STTC / CD / HYP) with official folds 1–8 / 9 / 10. Target band from Table 2: **macro-AUROC 0.92–0.93**.

In [ ]:
quick_flag = ["--quick"] if QUICK else []
cmd = ["python", "scripts/run_milestone2.py", "--config", "configs/runtime.yaml", "--output-dir", DRIVE_RESULTS, *quick_flag]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO))

## Milestone 3

1. Train a PTB-XL source model on the 10 shared SNOMED labels  
2. Unadapted OOD evaluation on Chapman and MIMIC (bootstrap CIs + chance test)  
3. Temperature Scaling and Linear Probing at P ∈ {5, 10, 20, 30, 50}% with 10 Monte Carlo reps

In [ ]:
targets = []
if "chapman" in roots:
    targets.append("chapman")
if "mimic" in roots:
    targets.append("mimic")
if not targets:
    raise RuntimeError("Need Chapman and/or MIMIC for Milestone 3")
print("M3 targets:", targets)

cmd = [
    "python", "scripts/run_milestone3.py",
    "--config", "configs/runtime.yaml",
    "--skip-m2",
    "--output-dir", DRIVE_RESULTS,
    "--targets", *targets,
]
if QUICK:
    cmd.append("--quick")
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO))

In [ ]:
!python scripts/summarize_results.py --output-dir $DRIVE_RESULTS

from pathlib import Path
import pandas as pd
adapt = Path(DRIVE_RESULTS) / "adaptation"
for csv in sorted(adapt.glob("*_pmin.csv")):
    print("\n", csv.name)
    display(pd.read_csv(csv))